### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="kdd_cup_09_appetency",
    dataset_year="2008",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://kdd.org/kdd-cup/view/kdd-cup-2009/Data",
    download_description="""
We download the data and the labels from the KDD challenge website:

mkdir -p local-data-warehouse/kdd_cup_09_appetency && \
wget -P local-data-warehouse/kdd_cup_09_appetency https://kdd.org/cupfiles/KDDCupData/2009/orange_small_train.data.zip && \
unzip local-data-warehouse/kdd_cup_09_appetency/orange_small_train.data.zip -d local-data-warehouse/kdd_cup_09_appetency/ && \
rm local-data-warehouse/kdd_cup_09_appetency/orange_small_train.data.zip && \
wget -P local-data-warehouse/kdd_cup_09_appetency https://kdd.org/cupfiles/KDDCupData/2009/orange_small_train_appetency.labels
""",
    # References
    academic_reference_bibtex=r"""@inproceedings{guyon2009analysis,
  title={Analysis of the kdd cup 2009: Fast scoring on a large orange customer database},
  author={Guyon, Isabelle and Lemaire, Vincent and Boull{\'e}, Marc and Dror, Gideon and Vogel, David},
  booktitle={KDD-Cup 2009 Competition},
  pages={1--22},
  year={2009},
  organization={PMLR}
}
""",
    academic_reference_bibtex_key="guyon2009analysis",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We use the small training data from the original data (230 features).
- We use appetency as label.
- We dropped empty columns: "Var8", "Var15", "Var20", "Var31", "Var32", "Var39", "Var42", "Var48", "Var52", "Var55", "Var79", "Var141", "Var167", "Var169", "Var175", "Var185", "Var209", "Var230".
- Anomaly: the feature names and categorical feature values have no semantic meaning.
- Anomaly: this dataset has many missing values.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Appetency",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Appetency",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "orange_small_train.data", sep="\t")
df_y = pd.read_csv(dataset_mold.path / "orange_small_train_appetency.labels", header=None)

target_feature = "Appetency"
df[target_feature] = df_y[0].astype("category")

empty_cols = [
    "Var8","Var15","Var20","Var31","Var32","Var39","Var42","Var48","Var52",
    "Var55","Var79","Var141","Var167","Var169","Var175","Var185","Var209","Var230",
]
df.drop(columns=empty_cols, inplace=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

cat_cols = [
    "Var192","Var193","Var194","Var195","Var196","Var197","Var198","Var199","Var200",
    "Var201","Var202","Var203","Var204","Var205","Var206","Var207","Var208","Var210",
    "Var211","Var212","Var214","Var216","Var217","Var218","Var219","Var220","Var222",
    "Var223","Var225","Var226","Var227","Var228","Var229",
    'Var191', 'Var213', 'Var215', 'Var221', 'Var224'
]
df[cat_cols] = df[cat_cols].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 50,000
Columns: 213
Use sampling: False (sample size: 50,000)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Var113', 'Var81', 'Var133', 'Var153', 'Var134', 'Var38', 'Var76', 'Var57', 'Var163', 'Var94']
Rows remaining as candidates after top-10 filter: 4 (of 50,000)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Var215,category,49306.0,98.61,1.0,eGzu
1,Var224,category,49180.0,98.36,1.0,4n2X
2,Var191,category,48917.0,97.83,1.0,r__I
3,Var213,category,48871.0,97.74,1.0,KdSa
4,Var194,category,37216.0,74.43,3.0,"SEuy, lvza, CTUH"
5,Var201,category,37217.0,74.43,2.0,"smXZ, 6dX3"
6,Var229,category,28432.0,56.86,4.0,"am7c, mj86, sk2h, oJmt"
7,Var225,category,26144.0,52.29,3.0,"ELof, kG3k, xG3x"
8,Var200,category,25408.0,50.82,15415.0,"yP09M03, Uw6SDm8, Ipi9M03, EvCZGt8, MF5S0rA, 5YIkUea, b1M9M03, NvI9wLk, gWmZGt8, Uw6kiXL"
9,Var214,category,25408.0,50.82,15415.0,"5zARyjR, PXYoFMh, 5zA8Zov, PXYT2rL, 5zAF4eX, Fhz3CDZ, PXYoYU2, 5zA2v9l, PXYGJt6, PXYwoeV"


In [5]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Var1,702.0,1.148718e+01,4.070995e+01,0.000000e+00,6.800000e+02
Var2,1241.0,4.029009e-03,1.419332e-01,0.000000e+00,5.000000e+00
Var3,1240.0,4.252984e+02,4.270194e+03,0.000000e+00,1.306680e+05
Var4,1579.0,1.253958e-01,1.275481e+00,0.000000e+00,2.700000e+01
Var5,1487.0,2.387933e+05,6.441259e+05,0.000000e+00,6.048550e+06
Var6,44471.0,1.326437e+03,2.685694e+03,0.000000e+00,1.317610e+05
Var7,44461.0,6.809496e+00,6.326053e+00,0.000000e+00,1.400000e+02
Var9,702.0,4.814530e+01,1.547779e+02,0.000000e+00,2.300000e+03
Var10,1487.0,3.926057e+05,9.280896e+05,0.000000e+00,1.232559e+07
Var11,1240.0,8.625806e+00,2.869558e+00,8.000000e+00,4.000000e+01


In [6]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column    rank                                         
Appetency 1                            -1  49110  98.22
          2                             1    890   1.78
Var191    1                          <NA>  48917  97.83
          2                          r__I   1083   2.17
Var192    1                    qFpmfo8zhV    385   0.77
          2                    DHeq9ayfAo    384   0.77
          3                    zKnr4RXktW    380   0.76
          4                    8I1r4RXXnK    379   0.76
          5                    HYTrjIK12c    379   0.76
Var193    1                          RO12  35964  71.93
          2                       2Knk1KF   7271  14.54
          3                       AERks4l   2243   4.49
          4                    g62hiBSaKg    580   1.16
          5                    e6CkoqApVR    524   1.05
Var194    1                          <NA>  37216  74.43
          2                          SEuy  12567  25.13
          3                          lvza    176   0.35
          4                          CTUH     41   0.08
Var195    1                          taul  47958  95.92
          2                    LfvqpCtLOY    866   1.73
          3              CiJDdr4TQ0rGERIS    544   1.09
          4                          ev6I    179   0.36
          5                       CuXi4je    159   0.32
Var196    1                          1K8T  49550  99.10
          2                          z3mO    432   0.86
          3                          JA1C     17   0.03
          4                          mKeq      1   0.00
Var197    1                          0Xwj   4629   9.26
          2                          lK27   4470   8.94
          3                          TyGl   4185   8.37
          4                          487l   3577   7.15
          5                          JLbT   2978   5.96
Var198    1                       fhk21Ss   4441   8.88
          2                       PHNvXy8   1150   2.30
          3                       iJzviRg    776   1.55
          4                       9GJGgoz    748   1.50
          5                       6CXYbuk    573   1.15
Var199    1                       r83_sZi    955   1.91
          2                    _jTP8ioIlJ    920   1.84
          3                       FoJylxy    762   1.52
          4                    76j2P_OLn0    652   1.30
          5                    glRBFJT8NN    597   1.19
Var200    1                          <NA>  25408  50.82
          2                       yP09M03     73   0.15
          3                       Uw6SDm8     48   0.10
          4                       Ipi9M03     45   0.09
          5                       EvCZGt8     34   0.07
Var201    1                          <NA>  37217  74.43
          2                          smXZ  12777  25.55
          3                          6dX3      6   0.01
Var202    1                          nyZz    198   0.40
          2                          VNjO    138   0.28
          3                          85IW    130   0.26
          4                          rlx_    128   0.26
          5                          gMVu    121   0.24
Var203    1                          9_Y1  45233  90.47
          2                          HLqf   3168   6.34
          3                          F3hy   1451   2.90
          4                          <NA>    143   0.29
          5                          dgxZ      4   0.01
Var204    1                          RVjC   1819   3.64
          2                          k13i   1658   3.32
          3                          m_h1   1289   2.58
          4                          7WNq   1096   2.19
          5                          SkZj   1091   2.18
Var205    1                          VpdQ  31962  63.92
          2                          09_Q  11574  23.15
          3                       sJzTlal   4530   9.06
          4                          <NA>   1934   3.87
Var206    1                          IYzP  17274  34.55
    

In [7]:
# Target Distribution
target_df

,count,pct
Appetency,,
-1,49110,98.22
1,890,1.78


## Task Curation

In [8]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to kdd_cup_09_appetency/019d7368-96e4-777a-bfbb-4fcb8602465c


019d7368-96e4-777a-bfbb-4fcb8602465c
980a8a08d61d926cb34ecc188f4e5e91e10af16e10dfaf41604a528e03b29b56
